In [ ]:
# LLM Agents in Google ADK: Packing List & Itinerary Example
# Welcome! In this notebook, you'll install Google’s Agent Development Kit (ADK), 
# configure your environment, define two LLM agents, and run your agent interactions—all in just a few steps.
# You'll see how LLM Agents power real-world travel apps like WanderWise, using natural language understanding, 
# reasoning, and tool integration to deliver smart, actionable results.

In [2]:
# Install Google ADK for Python
%pip install google-adk --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# (optional) Verify installation
%pip show google-adk

Name: google-adk
Version: 1.0.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: authlib, click, fastapi, google-api-python-client, google-cloud-aiplatform, google-cloud-secret-manager, google-cloud-speech, google-cloud-storage, google-genai, graphviz, mcp, opentelemetry-api, opentelemetry-exporter-gcp-trace, opentelemetry-sdk, pydantic, python-dotenv, PyYAML, sqlalchemy, tzlocal, uvicorn
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Configure environment
import os
# We use Gemini as our language model and ensure we are not using Vertex AI for this demo.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

In [5]:
# Import Required Modules
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from IPython.display import Markdown, display

In [14]:

# Packing List Agent: Definition
# The instruction set is the heart of the agent. Make it specific, structured, and safe.
packing_list_agent = Agent(
    name="packing_list_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="An agent that generates a packing list for a trip given a destination, duration, and user preferences.",
    instruction="""
        You are a helpful travel assistant.
        When the user provides a destination and trip duration, generate a detailed packing list tailored for that location and timeframe.
        Ask about planned activities and any special preferences or needs (such as business, hiking, or traveling with children).
        If the user wishes, you may ask about personal preferences (such as specific clothing or toiletries), but do not assume based on gender.
        Make sure your suggestions are practical and organized by category (clothing, toiletries, electronics, documents, etc.).
        If the user provides additional details, refine the list accordingly.
        Present the packing list in a clear and easy-to-read markdown format.
    """,
)
# NOTE: In future lessons, you will integrate external tools (e.g., weather APIs) for even more personalized lists.

In [15]:
# Packing List Agent: Session Setup and Run
# Define the app name, user ID, session ID, and user input for the packing list.
APP_NAME = "wanderwise_app"
USER_ID = "user_001"
SESSION_ID = "wanderwise_session_001"
PACKING_LIST_USER_INPUT = "I’m a 29F, and I am traveling to Tokyo for 5 days in April. I plan to do some easy hiking and some city sightseeing."

# Create a session service to manage user sessions.
session_service = InMemorySessionService()

# Create a session (async).
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# Create a content object that contains the user's request.
content = types.Content(role="user", parts=[types.Part(text=PACKING_LIST_USER_INPUT)])

# Create the runner for your packing list agent.
runner = Runner(
    agent=packing_list_agent,
    app_name=APP_NAME,
    session_service=session_service
)

# Invoke the agent with the user's input and print the response.
events = runner.run(
    user_id=USER_ID,
    session_id=SESSION_ID,
    new_message=content
)

for event in events:
    if event.is_final_response():
        response_text = event.content.parts[0].text
        display(Markdown(f"### Packing List\n\n{response_text}"))


### Packing List

Okay, I can help you create a detailed packing list for your 5-day trip to Tokyo in April, keeping in mind your plans for easy hiking and city sightseeing.

**First, let's get a bit more specific:**

*   **What kind of weather are you expecting in Tokyo in April?** (This will greatly influence clothing choices)
*   **Are there any specific hiking trails or areas you plan to visit?** (This helps determine the type of hiking gear needed)
*   **Are you planning any fancy dinners or events that would require dressier clothes?**
*   **Are there any personal preferences for clothing or toiletries you would like me to include?** (e.g. specific brands, favorite travel items, etc.)

Once I have this information, I can create a more tailored packing list.

**In the meantime, here is a general packing list to get you started:**

### **Clothing**

*   [ ] **Tops:** 5-7 (mix of short-sleeved and long-sleeved shirts for layering)
*   [ ] **Bottoms:** 2-3 pairs of pants/jeans, 1 pair of comfortable hiking pants or leggings
*   [ ] **Sweater/Jacket:** 1 lightweight fleece or sweater, 1 waterproof and windproof jacket
*   [ ] **Underwear:** 5-7 pairs
*   [ ] **Socks:** 5-7 pairs (including moisture-wicking socks for hiking)
*   [ ] **Sleepwear:** 1 set
*   [ ] **Comfortable Walking Shoes:** 1 pair (broken in!)
*   [ ] **Hiking Shoes/Boots:** 1 pair (if you plan on more than just very light trails)
*   [ ] **Sandals or Flats:** 1 pair (for evenings or indoor use)
*   [ ] **Accessories:**
    *   [ ] Scarf or bandana
    *   [ ] Hat or cap
    *   [ ] Gloves (especially if going early April)

### **Toiletries**

*   [ ] **Travel-sized toiletries:**
    *   [ ] Shampoo
    *   [ ] Conditioner
    *   [ ] Body wash
    *   [ ] Toothpaste
    *   [ ] Toothbrush
    *   [ ] Deodorant
*   [ ] **Skincare:**
    *   [ ] Sunscreen
    *   [ ] Face moisturizer
    *   [ ] Lip balm with SPF
*   [ ] **Makeup:** (as desired, keep it minimal)
*   [ ] **Other:**
    *   [ ] Insect repellent
    *   [ ] Hand sanitizer
    *   [ ] Wet wipes
    *   [ ] Feminine hygiene products

### **Electronics**

*   [ ] **Phone & Charger**
*   [ ] **Portable Charger/Power Bank**
*   [ ] **Camera & Charger** (optional)
*   [ ] **Travel Adapter** (Japan uses Type A and B plugs, 100V)
*   [ ] **Headphones**

### **Documents & Essentials**

*   [ ] **Passport & Visa** (if required)
*   [ ] **Flight/Hotel Confirmations**
*   [ ] **Copies of Important Documents** (keep separately)
*   [ ] **Credit Cards & Cash** (inform your bank of travel dates)
*   [ ] **Japan Rail Pass** (if purchased)
*   [ ] **Phrasebook or Translation App**
*   [ ] **Travel Insurance Information**

### **Health & Safety**

*   [ ] **First-Aid Kit** (band-aids, pain relievers, antiseptic wipes, etc.)
*   [ ] **Prescription Medications** (with copies of prescriptions)
*   [ ] **Motion Sickness Remedies** (if needed)

### **Miscellaneous**

*   [ ] **Reusable Water Bottle**
*   [ ] **Small Backpack or Daypack**
*   [ ] **Reusable Shopping Bag**
*   [ ] **Snacks** (for hikes and long days)
*   [ ] **Eye Mask & Earplugs**
*   [ ] **Book or Entertainment**

**Once you provide me with the additional details, I can refine this list further to make it perfect for your trip!**


In [16]:
# 🎉 Congratulations!
# You’ve just built and run your first LLM agent!
# You defined an agent, created a session, and ran it with a runner—the building blocks for more advanced multi-agent systems.
# More details about sessions and runners are coming up—stay tuned!

In [17]:
# Itinerary Agent with Google Search Tool 
# Now let's go a step further: create an itinerary agent that uses the [google_search] tool!
# This agent will generate a personalized itinerary for your trip, leveraging live search results.

In [18]:
# Import the Google Search tool from ADK
from google.adk.tools import google_search

In [19]:
# Itinerary Agent: Definition
# The agent's instructions tell it to use the google_search tool to find up-to-date recommendations.
itinerary_agent = Agent(
    name="itinerary_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="An agent that creates a travel itinerary for a given destination and trip duration, using Google Search for up-to-date recommendations.",
    instruction="""
        You are a helpful and creative travel itinerary planner.
        When the user provides a destination and trip duration, use the [google_search] tool to find current and popular attractions, restaurants, and activities for each day.
        For each day, create a detailed schedule with 2-4 activities, including at least one meal suggestion and one local attraction, prioritizing the user's interests (such as art and food).
        Present the itinerary in a clear, easy-to-read markdown format, organized by day.
        If you need more information, ask the user for preferences or interests, but if you have enough, proceed to generate a full itinerary.
        Example format:

        ### Day 1
        - Morning: [Attraction or activity]
        - Lunch: [Restaurant or food experience]
        - Afternoon: [Attraction or activity]
        - Evening: [Dinner suggestion or event]

        ### Day 2
        ...

        Always use the [google_search] tool to find the latest recommendations for each activity or restaurant.
        Be concise, friendly, and ensure the itinerary is complete for all days requested.
    """,
    tools=[google_search],
)

In [20]:
# Itinerary Agent: Session Setup and Run
ITINERARY_USER_INPUT = "I'm visiting Barcelona for 3 days and I love art and food. Can you plan my trip?"

# (Re-use the same session service, or create a new one if needed)
# We'll use a new session ID for the itinerary agent.
ITINERARY_SESSION_ID = "wanderwise_itinerary_session_001"

await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=ITINERARY_SESSION_ID
)

# Create a content object for the itinerary request
itinerary_content = types.Content(role="user", parts=[types.Part(text=ITINERARY_USER_INPUT)])

# Create a runner for the itinerary agent
itinerary_runner = Runner(
    agent=itinerary_agent,
    app_name=APP_NAME,
    session_service=session_service
)

# Run the itinerary agent and display the response in markdown
itinerary_events = itinerary_runner.run(
    user_id=USER_ID,
    session_id=ITINERARY_SESSION_ID,
    new_message=itinerary_content
)

for event in itinerary_events:
    if event.is_final_response():
        # Concatenate all parts in the final response
        full_text = ''.join(part.text for part in event.content.parts if part.text)
        display(Markdown(f"### Your Itinerary\n\n{full_text}"))


### Your Itinerary

Okay, I can help you plan your 3-day trip to Barcelona, focusing on art and food. Here's a possible itinerary:

Okay, here's a possible 3-day itinerary for your trip to Barcelona, focusing on art and food:

### Day 1: Gothic Quarter & Picasso
*   **Morning:** Start with a visit to the Museu Picasso. This museum houses an extensive collection of Picasso's early works and reveals his relationship with the city. (Metro: Jaume I). Consider booking tickets in advance to avoid lines.
*   **Lunch:** Bar del Pla, near the Picasso Museum, is known for its authentic tapas and Catalan dishes made with locally sourced ingredients.
*   **Afternoon:** Explore the Gothic Quarter. Wander through the narrow, winding streets, visit the Barcelona Cathedral, and soak in the historic atmosphere.
*   **Evening:** Enjoy dinner at 7 Portes, a classic restaurant in the El Born neighborhood, known for its traditional Catalan cuisine, especially paella.

### Day 2: Gaudí & Catalan Modernism
*   **Morning:** Visit the Sagrada Familia, Gaudí's iconic basilica. Book your tickets online in advance to secure a time slot and avoid long queues.
*   **Lunch:** The Rooftop at Sir Victor offers stunning views of the Sagrada Familia and Casa Milà, along with light bites, tapas, and cocktails.
*   **Afternoon:** Explore Park Güell, another of Gaudí's masterpieces, offering unique architecture and panoramic city views. Purchase tickets in advance.
*   **Evening:** Experience a food tour in the El Born or Gothic Quarter. Several companies, like "The Barcelona Taste" or "Secret Food Tours," offer small group tours that introduce you to the best tapas and local specialties.

### Day 3: Art, Architecture & Culinary Delights
*   **Morning:** Visit Casa Batlló or Casa Milà (La Pedrera), two of Gaudí's most famous houses on Passeig de Gràcia. Choose one based on your preference for their unique architectural styles.
*   **Lunch:** Bar Mut is a highly-rated restaurant offering a casual atmosphere with haute cuisine.
*   **Afternoon:** Explore the Museu Nacional d'Art de Catalunya (MNAC), which houses a comprehensive collection of Catalan art from Romanesque to modern times. It is located in the Palau Nacional on Montjuïc.
*   **Evening:** Indulge in a memorable dining experience at Alkimia, blending traditional Catalan recipes with avant-garde techniques. Alternatively, Disfrutar, led by former El Bulli chefs, offers a creative and innovative tasting menu.

Enjoy your art and food-filled adventure in Barcelona!


In [21]:
# 🎉 Congratulations!
# You’ve now built and run two different LLM agents in ADK:
# - A packing list agent for smart, personalized packing suggestions
# - An itinerary agent that uses live Google Search results to plan an amazing trip
# This is the foundation for building rich, multi-agent travel assistants with Google ADK.